# Stage 1 at the NEOARM: recording how beam shift and tilt displace the beam on the camera

This notebook drives the microscope through PyJEM. At 22 projection-lens settings around the 40k
MAG row and the 40 cm DIFF row it deflects the beam with CLA1 (shift) and CLA2 (tilt), + and −,
along x and y, and saves every camera frame with a log line. Nothing is fitted here: copy the
output folder to a laptop and fit it there with `stage1_rehearsal.ipynb` (`fit_response(folder)`).

Every cell that changes the microscope is first run as a **dry run**: it reads the microscope but
changes nothing, and prints each change it would make. It needs only numpy and PyJEM, and
`../pyjem_session.py` next to this folder.

In [ ]:
import sys

import numpy as np

sys.path.append("..")
from pyjem_session import Session, check_pyjem, newest_file_in   # the session tools (numpy and PyJEM only)

## 1. Connecting to the microscope

PyJEM gives `TEM3` (lenses, deflectors, modes) and `detector` (the camera). If the camera is not
reachable through PyJEM, use the camera's own software to save every frame into a folder and set
`take_frame = newest_file_in("D:/camera_frames")`: each frame is then the newest file there.

In [ ]:
from PyJEM import TEM3

try:
    from PyJEM import detector
except ImportError:
    detector = None
    print("this PyJEM has no detector module: set take_frame = newest_file_in(...) below")

take_frame = None                                        # None: frames from PyJEM's camera

## 2. Checking the camera and the beam before changing anything

First, which of the PyJEM calls the session uses this PyJEM has (an older PyJEM may lack some).
This changes nothing.

In [ ]:
missing = check_pyjem(TEM3=TEM3, detector=detector)

Then the camera on its own, straight through PyJEM: its settings, and the size of one raw frame
(2 bytes per pixel for a 16-bit camera, 4 for 32-bit).

In [ ]:
if take_frame is None:
    camera = detector.Detector(detector.get_attached_detector()[0])
    print(camera.get_detectorsetting())
    print(len(camera.snapshot_rawdata()), "bytes in one frame")
else:
    print("frames come from take_frame, not through PyJEM")

Then the session. Opening it changes nothing: it notes the microscope's state, so that
`session.restore()` can put it back. **The session is a dry run unless a block explicitly allows
changes**: anywhere outside a `with session.dry_run(False):` block, whatever it is asked to do is
only printed ("would set IL1 21534 -> 23534"). Here it takes one frame and finds the beam.

In [ ]:
LENSES = ("IL1", "IL2", "IL3", "PL1")

session = Session(TEM3=TEM3, detector=detector, take_frame=take_frame, lenses=LENSES, settle_s=2,
                  folder="output/stage1_neoarm/projection_response")

beam = session.beam()
print(f"beam at ({beam.centre.real:+.1f}, {beam.centre.imag:+.1f}) px from the centre, rms {beam.rms:.1f} px")
print("problems:", "; ".join(beam.problems) or "none")

Every cell below that changes the microscope looks like this:

```python
dry_run = True           # True: only print what this cell would do. False: do it.

with session.dry_run(dry_run):
    ...
```

Run it with `True`, read what it prints, then set `False` and run it again. After the block the
session is a dry run again, whatever happened inside. In a dry run nothing changes, so any
displacements it prints are not real; the "would ..." lines are what matters. The cells
are saved with `True`, so running the notebook as it is changes nothing; with nothing recorded,
the fit and the figures just say so.

## 3. Reading the lens settings of the 40k MAG row and the 40 cm DIFF row

The experiment works around two starting points: the 40k MAG row (where the microscope is now) and
the DIFF row at 40 cm (IL1–PL1 as the console sets them in DIFF; a dry run cannot know those, it
does not switch). Then IL1–PL1 go under free-lens control at the 40k values, and a dark frame is
taken with the beam blanked.

In [ ]:
dry_run = True           # True: only print what this cell would do. False: do it.

lenses_at = {}                                   # IL1-PL1 DAC values at each starting point

with session.dry_run(dry_run):
    lenses_at["40k"] = session.lenses_now()
    session.diffraction(camera_length_cm=40)
    lenses_at["DIFF"] = session.lenses_now()
    session.imaging()

    session.take_over_lenses()
    session.dark_frame()

lenses_at

## 4. Choosing the deflection sizes for CLA1 (shift) and CLA2 (tilt)

At each starting point, each deflector is deflected by 100 units, then 200, 400, ... until it
displaces the beam by more than 25 px (at most 20 000 units). If the beam leaves the camera, the deflection goes back to
half. These deflection sizes are used, along x and along y, for every lens setting around that starting point.

In [ ]:
dry_run = True           # True: only print what this cell would do. False: do it.

deflection_units = {}          # deflection size, per starting point and deflector
displacement_px = {}           # how far that deflection displaces the beam on the camera (x + iy)

with session.dry_run(dry_run):
    for start in ["40k", "DIFF"]:
        session.set_lenses(lenses_at[start])
        beam_not_deflected = session.beam(role="trying a deflection")

        for deflector in ["CLA1", "CLA2"]:
            print(f"{start}, {deflector}: deflections of 100, 200, 400, ... units (at most 20000) until the beam is displaced by > 25 px")
            deflection = 100
            while True:
                session.deflect(deflector, deflection)
                beam_deflected = session.beam(role="trying a deflection")
                session.deflect(deflector, 0)
                displacement = beam_deflected.centre - beam_not_deflected.centre
                print(f"   {deflector} {deflection:5d} units: beam displaced {abs(displacement):5.1f} px")

                if not beam_deflected.usable:           # off the camera: back to half
                    deflection = deflection // 2
                    break
                if abs(displacement) > 25 or deflection == 20000:
                    break
                deflection = min(2 * deflection, 20000)

            deflection_units[start, deflector] = deflection
            displacement_px[start, deflector] = displacement

deflection_units

## 5. Choosing the step sizes for IL1–IL3 and PL1

At each starting point, each lens is stepped up by 250 DAC units, then 500, 1000, ... until the
step clearly changes the displacement from that starting point's stronger deflection (by more than 10 px),
at most 8000 units and never past 2000 or 65535. The step found is then used up and down.

In [ ]:
dry_run = True           # True: only print what this cell would do. False: do it.

lens_step_units = {}                             # lens step, per starting point and lens

with session.dry_run(dry_run):
    for start in ["40k", "DIFF"]:
        # the deflector whose deflection displaces the beam more here is the yardstick
        if abs(displacement_px[start, "CLA1"]) > abs(displacement_px[start, "CLA2"]):
            deflector = "CLA1"
        else:
            deflector = "CLA2"
        deflection = deflection_units[start, deflector]
        displacement_at_start = displacement_px[start, deflector]

        for i, lens in enumerate(LENSES):
            largest_step = int(min(8000, 65535 - lenses_at[start][i], lenses_at[start][i] - 2000))
            print(f"{start}, {lens}: steps of 250, 500, 1000, ... DAC units (at most {largest_step}) "
                  f"until the {deflector} deflection's displacement changes by > 10 px")
            lens_step = 250
            while True:
                lenses = lenses_at[start].copy()
                lenses[i] = lenses[i] + lens_step
                session.set_lenses(lenses)
                beam_not_deflected = session.beam(role="trying a lens step")
                session.deflect(deflector, deflection)
                beam_deflected = session.beam(role="trying a lens step")
                session.deflect(deflector, 0)
                displacement = beam_deflected.centre - beam_not_deflected.centre
                change = abs(displacement - displacement_at_start)
                print(f"   {lens} +{lens_step:4d}: the {deflector} deflection's displacement changed by {change:5.1f} px")

                if not (beam_not_deflected.usable and beam_deflected.usable):     # beam lost: back to half
                    lens_step = lens_step // 2
                    break
                if change > 10 or lens_step >= largest_step:
                    break
                lens_step = min(2 * lens_step, largest_step)

            lens_step_units[start, lens] = lens_step

        session.set_lenses(lenses_at[start])

lens_step_units

## 6. Measurement plan: lens settings and deflections

Every lens setting to be measured, and how far each deflector will be deflected there (+ and −, along x
and along y).
Around each starting point: the starting point itself, each lens stepped up and down by its step,
and the starting point again; all with the deflection sizes found for that starting point. Then
the 40k row once more. The repeats of the 40k row give the wobble.

In [ ]:
measurement_plan = []

for start in ["40k", "DIFF"]:
    deflections = {"CLA1": deflection_units[start, "CLA1"], "CLA2": deflection_units[start, "CLA2"]}

    measurement_plan.append(dict(name=f"{start} row", lenses=lenses_at[start], deflections=deflections))
    for i, lens in enumerate(LENSES):
        for sign in [+1, -1]:
            lenses = lenses_at[start].copy()
            lenses[i] = lenses[i] + sign * lens_step_units[start, lens]
            name = f"{start} row: {lens} {sign * lens_step_units[start, lens]:+d}"
            measurement_plan.append(dict(name=name, lenses=lenses, deflections=deflections))
    measurement_plan.append(dict(name=f"{start} row", lenses=lenses_at[start], deflections=deflections))

measurement_plan.append(dict(name="40k row", lenses=lenses_at["40k"],
                 deflections={"CLA1": deflection_units["40k", "CLA1"], "CLA2": deflection_units["40k", "CLA2"]}))

for number, setting in enumerate(measurement_plan, 1):
    lenses = ", ".join(f"{lens} {value:.0f}" for lens, value in zip(LENSES, setting["lenses"]))
    deflections = f"CLA1 ±{setting['deflections']['CLA1']}, CLA2 ±{setting['deflections']['CLA2']} (x and y)"
    print(f"{number:2d}. {setting['name']:22s} {lenses}    {deflections}")

## 7. Recording the beam displacements

At each lens setting of the measurement plan: a frame with no deflection, then each deflector deflected + and −
by the measurement plan's deflection, first along x and then along y, a frame each (9 frames). Every frame is saved.

In [ ]:
dry_run = True           # True: only print what this cell would do. False: do it.

with session.dry_run(dry_run):
    for number, setting in enumerate(measurement_plan, 1):
        print(f"{number}. {setting['name']}")
        about_this_setting = dict(number=number, setting=setting["name"], lenses=[float(v) for v in setting["lenses"]])

        session.set_lenses(setting["lenses"])
        session.frame(role="not deflected", **about_this_setting)

        for deflector in ["CLA1", "CLA2"]:                  # CLA1 shifts the beam, CLA2 tilts it
            size = setting["deflections"][deflector]
            for direction, deflection in [("x", size), ("y", 1j * size)]:      # deflector units, x + iy
                session.deflect(deflector, +deflection)
                session.frame(role="deflected +", deflector=deflector, direction=direction,
                              deflection_units=[deflection.real, deflection.imag], **about_this_setting)
                session.deflect(deflector, -deflection)
                session.frame(role="deflected -", deflector=deflector, direction=direction,
                              deflection_units=[-deflection.real, -deflection.imag], **about_this_setting)
            session.deflect(deflector, 0)

    session.restore()

## After the recording

Copy `output/stage1_neoarm/` to the laptop. There, in `stage1_rehearsal.ipynb`,
`model = fit_response("output/stage1_neoarm/projection_response")` fits the lens model and
`frames(...)` shows the frames.